# Non-pivot xslx functions 2

Created: 2026-09-30

Develop functions for accessing data in xlsx workbooks that do not contain pivot tables.

The previous notebook in this series [Non-pivot xlsx functions 1](non-pivot-xlsx-01.ipynb) worked through functionality for table 14-12-07 of the [December 2014 statistics for foreign residents in Japan](http://www.e-stat.go.jp/SG1/estat/List.do?lid=000001133760). This notebook works through other tables from that dataset for the main purpose of further developing the functionality for deriving column headers from the xlsx files.

The approach in this notebook worked for the cross tabulation format files in the December 2014 release but failed for other releases. See [Non-pivot xlsx functions 3](non-pivot-xlsx-03.ipynb) for a better approach that uses tree data structures for parsing the column headers and stub column.

## Install the necessary packages

In [ ]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install numpy
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install openpyxl

## Install the newest version of estatjp

Repeat during development to import revisions.

In [ ]:
# run the following in the terminal
python -m pip install -e .

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [ ]:
# 14-12-01-1 国籍・地域別　在留資格（在留目的）別　在留外国人
url14120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226524&fileKind=0" # 14-12-01-01 - Dec 2014 by nationality/region and residence class.
url14120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226525&fileKind=0" # 14-12-01-02 - Dec 2014 by nationality/region and residence class.
url14120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226526&fileKind=0" # 14-12-02-01 - Dec 2014 by nationality/region age and sex.
url14120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226527&fileKind=0" # 14-12-02-01 - Dec 2014 by nationality/region age and sex.
url141203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226528&fileKind=0" # 14-12-03 - Dec 2014 by residence class, age, sex
url141204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226529&fileKind=0" # 14-12-04 - Dec 2014 by prefecture, nationality/region
url14120500 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226530&fileKind=0" # 14-12-05-00 - Dec 2014 by prefecture, residence class. (total)
url14120501 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226531&fileKind=0" # 14-12-05-01 - Dec 2014 by prefecture, residence class. (China)
url14120502 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226612&fileKind=0" # 14-12-05-02 - Dec 2014 by prefecture, residence class. (Taiwan)
url14120503 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226613&fileKind=0" # 14-12-05-03 - Dec 2014 by prefecture, residence class. (Korea, NK)
url14120504 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226614&fileKind=0" # 14-12-05-04 - Dec 2014 by prefecture, residence class. (Phillipines)
url14120505 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226615&fileKind=0" # 14-12-05-05 - Dec 2014 by prefecture, residence class. (Brazil)
url14120600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226616&fileKind=0" # 14-12-06-00 - Dec 2014 by prefecture, age and sex. (total)
url14120601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226617&fileKind=0" # 14-12-06-01 - Dec 2014 by prefecture, age and sex. (China)
url14120602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226618&fileKind=0" # 14-12-06-02 - Dec 2014 by prefecture, age and sex. (Taiwan)
url14120603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226619&fileKind=0" # 14-12-06-03 - Dec 2014 by prefecture, age and sex. (Korea, NK)
url14120604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226620&fileKind=0" # 14-12-06-04 - Dec 2014 by prefecture, age and sex. (Phillipines)
url14120605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226621&fileKind=0" # 14-12-06-05 - Dec 2014 by prefecture, age and sex. (Brazil)
url141207 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226622&fileKind=0" # 14-12-07 - Dec 2014 by municipality, nationality/region


- url14120101 is a table that is included in the time series database that is accessed in the notebook [Accessing the MOJ/ISA data via API calls](moj-isa-via-api.ipynb). It has a decked header (right figure below) and a two-column stub that differentiates between region names and nation names by placing region names in column A with merging into column B, and nation names in column B (left figure below). Note that the title is in cell B2, not A2.

| ![Table 14-12-01-1 upper left corner](images/Table14-12-01-1-upper-left-corner.jpg) | ![Table 14-12-01-1 decked headers](images/Table14-12-01-1-decked-headers.jpg) |
|-------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
| Table 14-12-01-1 upper left corner                                                  | Table 14-12-01-1 decked headers                                               |

## Import the module and download the file

In [ ]:
from estatjp import excel
from pathlib import Path

ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa")
newdata = excel.download_to_folder(url14120101, dataraw, overwrite=True, verbose=True)
newdata
print('source: ' + str(newdata))
if newdata != None:
    processeddata = ROOT_DIR / Path(".data/processed/isa") / newdata.name
    processeddata = processeddata.with_suffix(".csv")
    print('target: ' + str(processeddata))

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Strict-Transport-Security': 'max-age=0', 'Content-Disposition': "attachment; filename*=UTF-8''14-12-01-1.xlsx", 'Accept-Ranges': 'bytes', 'X-Cache': 'MISS', 'Date': 'Sat, 17 Oct 2026 05:41:11 GMT', 'Cache-Control': 'private', 'X-Zen-Fury': '8dbff0dec214198cca26128ad35fade8c0d77416', 'Server': 'ZENEDGE', 'X-Cache-Status': 'NOTCACHED', 'Content-Encoding': 'gzip', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '1; mode=block', 'X-Frame-Options': 'SAMEORIGIN', 'X-Cdn': 'Served-By-Zenedge'}
source: c:\Users\Alan\Documents\GitHub\estatjp\.data\raw\isa\14-12-01-1.xlsx
target: c:\Users\Alan\Documents\GitHub\estatjp\.data\processed\isa\14-12-01-1.csv


## Examine the worksheets and look for pivot tables

If there are pivot tables, use the pivot module. Otherwise use the excel module. This notebook is only for reporting workbooks and doesn't use analytical pivot workbooks.

In [23]:
import openpyxl
wb = openpyxl.load_workbook(newdata, data_only=True)
snames = wb.sheetnames
print(snames)
has_pivot = False
has_pivot_cache_records = False
for ws in wb.worksheets:
    print('title: ' + ws.title)
    print('dimensions: ' + ws.dimensions)
    if ws.print_title_cols != None:
        print('print_title_cols: ' + ws.print_title_cols)
    if ws.print_title_rows != None:
        print('print_title_rows: ' + ws.print_title_rows)
    print('_pivots len: ' + str(len(ws._pivots)))
    if len(ws._pivots) > 0:
        has_pivot = True
        print('pivot_table.cache.records: ' + str(ws._pivots[0].cache.records))
        if ws._pivots[0].cache.records != None:
            has_pivot_cache_records = True
    print('merged_cells len: ' + str(len(ws.merged_cells.ranges)))


['14-12-01-1']
title: 14-12-01-1
dimensions: A2:AS207
print_title_cols: $A:$B
print_title_rows: $4:$6
_pivots len: 0
merged_cells len: 46


### Use a [regular expression](https://docs.python.org/3/library/re.html) to identify and parse the title

Use an actual example to develop the regular expression. The group names 'cat01' and 'cat02' match those used by e-Stat's API service (See [Accessing the MOJ/ISA data via API calls](moj-isa-via-api.ipynb)).

In [24]:
import re
TITLE_REGEX = re.compile(r"第\d+表\S*\s+(?P<cat01_label>\S+)別\s+(?P<cat02_label>\S+)別\s+(?P<title_population>\w+)\s*(（(その\w+\s+)?(?P<sub_population>[\s\S]+)）)?")
title_str = "第１表　国籍・地域別　在留資格（在留目的）別　在留外国人" 
m= TITLE_REGEX.match(title_str)
print(m)
if m != None:
    print(m.group('cat01_label'))
    print(m.group('cat02_label'))
    print(m.group('title_population'))
    print(m.group('sub_population'))
else:
    print('irregular title: ' + title_str)

<re.Match object; span=(0, 28), match='第１表\u3000国籍・地域別\u3000在留資格（在留目的）別\u3000在留外国人'>
国籍・地域
在留資格（在留目的）
在留外国人
None


## Parse the print title rows to get the title and the column headers.

- url14120101 The print title rows run from 4 to 6. In the previous url14120101, the print rows were marked as 1 through 4, but the column headers were only in row 4.

    Here treat rows 1-3 as title rows, and 4 through the rest of the print_title_rows.

    url14120101 also has a PrintTitles.cols member that can be used for parsing row titles.

    Neither PrintTitles.rows nor PrintTitles.cols are guaranteed to be present, but for now use them.

### Locate stub heading

The stub heading is at the top left corner of the table and describes the leftmost column. Try to locate it and its determine its width and height using merged cell ranges. 

- url14120201 Has dimensions larger than the table inside. We need a more reliable method to determine the bounding box. Use the cells.


In [25]:
def get_cells_bounding_box(cells):
    """
    Given an iterable of openpyxl Cell objects, return the bounding box.
    Returns: (min_row, max_row, min_col, max_col)
    """
    if not cells:
        raise ValueError("No cells provided.")

    min_row = min(cell.row for cell in cells)
    max_row = max(cell.row for cell in cells)
    min_col = min(cell.column for cell in cells)
    max_col = max(cell.column for cell in cells)

    return min_row, max_row, min_col, max_col

## Scan the worksheet to isolate tables

Worksheet dimensions are not reliable for determining the columns and rows of the table. Some worksheets have blank columns left and right, and some have blank rows below the table.

Eventually this functionality will be extended to handle worksheets with multiple tables. The worksheets in the target dataset only have one table each.

In [ ]:
from openpyxl import Workbook
from openpyxl.cell.cell import Cell
from openpyxl.cell.cell import MergedCell
from openpyxl.worksheet import merge
from openpyxl.worksheet.cell_range import (MultiCellRange, CellRange)
import openpyxl.utils.cell
from openpyxl.utils import (
    get_column_letter,
)

ws = wb.active # There is only one worksheet in the current workbook
# _print_rows and _print_cols may be useful if present. But unreliable.
print(ws.print_title_rows)
print(ws.print_titles)
if ws._print_rows != None:
    print(ws._print_rows.min_row)
    print(ws._print_rows.max_row)

coord = ws.dimensions
stated_range = CellRange(coord)
stated_left_col = get_column_letter(stated_range.min_col)
print('Stated dimensions: ' + str(coord))

# look for the first row with the rightmost cell in the maximum column
bounds = openpyxl.utils.cell.range_boundaries(coord)
print(bounds)
stated_min_col = bounds[0]
stated_min_row = bounds[1]
stated_max_col = bounds[2]
stated_max_row = bounds[3]
found_max_col = stated_min_col

valcellrange = None

rowiter = ws.iter_rows(min_row=stated_min_row, max_row=stated_max_row, min_col = stated_min_col, max_col=stated_max_col)
#rows = openpyxl.utils.rows_from_range(coord)
for row in rowiter:
    hitcount = 0
    hitcellrange = None
    blankcellrange = None
    blankcount = 0
    max_val_col = 0
    for idxcell, cell in enumerate(row):
        if (isinstance(cell, Cell) and cell.value != None):
            hitcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(hitcellrange, CellRange):
                hitcellrange = hitcellrange.union(thiscellrange)
            else:
                hitcellrange = thiscellrange
            if idxcell+1 > max_val_col:
                max_val_col = cell.column
                if max_val_col > found_max_col:
                    found_max_col = max_val_col
        elif (isinstance(cell, Cell) and cell.value == None) or isinstance(cell, MergedCell):
            blankcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(blankcellrange, CellRange):
                blankcellrange = blankcellrange.union(thiscellrange)
            else:
                blankcellrange = thiscellrange
    if isinstance(valcellrange, CellRange) and isinstance(hitcellrange, CellRange):
        valcellrange = valcellrange.union(hitcellrange)
    elif isinstance(hitcellrange, CellRange):
        valcellrange = hitcellrange
    print(str(hitcount) + ' values out of ' + str(len(row)) + ' max column=' + str(max_val_col))
valbounds = valcellrange.bounds
print('valbounds: ' + str(valbounds))

$4:$6
'14-12-01-1'!$4:$6,'14-12-01-1'!$A:$B
4
6
Stated dimensions: A2:AS207
(1, 2, 45, 207)
1 values out of 45 max column=2
0 values out of 45 max column=0
30 values out of 45 max column=45
10 values out of 45 max column=40
9 values out of 45 max column=39
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max column=45
44 values out of 45 max colum

## Locate stub cell

In a properly formed table, the stub cell is the top left cell of the table. It may be merged with cells to the right for multilevel row labels, and with cells below for decked column headers. In the functions below, the stub cell range is essential for determining the cat01 and cat02 columns in the column-oriented output.

In [29]:
from openpyxl.utils.cell import (
    coordinate_from_string,
    coordinate_to_tuple,
)

stub_cell = stated_left_col + str(stated_range.min_row)
if ws._print_rows != None:
    stub_cell = stated_left_col + str(ws._print_rows.min_row) # This may be right but check it below
    
rowiter = ws.iter_rows(min_row=valbounds[1], max_row=valbounds[3], min_col = valbounds[0], max_col=valbounds[2])
#rows = openpyxl.utils.rows_from_range(coord)
for row in rowiter:
    hitcount = 0
    hitcellrange = None
    blankcellrange = None
    blankcount = 0
    max_val_col = 0
    for idxcell, cell in enumerate(row):
        if (isinstance(cell, Cell) and cell.value != None):
            hitcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(hitcellrange, CellRange):
                hitcellrange = hitcellrange.union(thiscellrange)
            else:
                hitcellrange = thiscellrange
            if idxcell+1 > max_val_col:
                max_val_col = cell.column
                if max_val_col > found_max_col:
                    found_max_col = max_val_col
        elif (isinstance(cell, Cell) and cell.value == None) or isinstance(cell, MergedCell):
            blankcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(blankcellrange, CellRange):
                blankcellrange = blankcellrange.union(thiscellrange)
            else:
                blankcellrange = thiscellrange
    print(str(hitcount) + ' values out of ' + str(len(row)) + ' max column=' + str(max_val_col))
    if hitcount > blankcount:
        stub_cell = openpyxl.utils.cell.get_column_letter(bounds[0]) + str(cell.row)
        break
# The above gives the top left cell, which is supposed to have a stub name in it. If there is no name, 
# move right one cell
stubtuple = coordinate_to_tuple(stub_cell)
sc = ws.cell(row=stubtuple[0], column=stubtuple[1])
if sc.value is None:
    sc.value = 'Stub'
print('stub cell: ' + stub_cell)
print('Found max column: ' + str(found_max_col))
# now find MergedCellRange that contains the stub_cell
stubcr = CellRange(range_string=stub_cell)
mcr = MultiCellRange(ws.merged_cells.ranges)
crfound = None
if mcr.__contains__(stub_cell):
    for cr in mcr.__iter__():
        if cr.__contains__(stub_cell):
            crfound = cr.__copy__()
            break
if isinstance(crfound, CellRange):
    stubcr = CellRange(range_string=crfound.coord)
print('Stub cell range: min_row= ' + str(stubcr.min_row)
      + ' max_row= ' + str(stubcr.max_row) 
      + ' min_col= ' + str(stubcr.min_col)  
      + ' max_col= ' + str(stubcr.max_col))

1 values out of 45 max column=2
0 values out of 45 max column=0
30 values out of 45 max column=45
stub cell: A4
Found max column: 45
Stub cell range: min_row= 4 max_row= 6 min_col= 1 max_col= 2


The stub cell range `stubcr` should be the most reliable method for determining the columns in the variable column set and the rows in the header row set. Use it below to parse the column headers and the row index labels.

### Carry functions to parse and duplicate header strings that span multiple columns or multiple rows 

In [30]:
# function to carry column members between the carried column and new column iteratively
def carry_column_members( carriedlist, newlist, index ):
    if newlist.count(None) == len(newlist):
        return carriedlist.copy()
    if carriedlist[index] != None and newlist[index] == None:
        newlist[index] = carriedlist[index]
        index = index + 1
        if index < len(newlist):
            carry_column_members(carriedlist, newlist, index)
    return newlist.copy()


In [31]:
# function to carry row members between the carried row and new row iteratively
def carry_row_members( carriedlist, newlist, index ):
    if newlist.count(None) == len(newlist):
        return carriedlist.copy()
    if carriedlist[index] != None and newlist[index] == None:
        newlist[index] = carriedlist[index]
        index = index + 1
        if index < len(newlist):
            carry_row_members(carriedlist, newlist, index)
    return newlist.copy()


## Parse the sheetname, the title information and the column headers

In [32]:
import re
import datetime as dt
from openpyxl import Workbook
from openpyxl.cell.cell import Cell
from openpyxl.worksheet import cell_range

SHEETNAME_REGEX = re.compile(r"(?P<year>\d{2})-(?P<month>\d{2})-(?P<tableno>\d{1,2}(-[\da-zA-Z0-9_]+)?)")
#TITLE_REGEX = re.compile(r"第\d+表\S*\s+(?P<cat01_label>\S+)別\s+(?P<cat02_label>\S+)別\s+(?P<title_population>\w+)(（(その\w+\s+)?(?P<sub_population>\S+)）)?")
ws = wb.active # There is only one worksheet in the current workbook
print(ws.print_title_rows)
print(ws.print_titles)
if ws._print_rows != None:
    print(ws._print_rows.min_row)
    print(ws._print_rows.max_row)
title_result = {}
# the year, month and table name are in ws.title
mt = SHEETNAME_REGEX.match(ws.title)
if mt != None:
    yr = int("20" + mt.group('year'))
    month = int(mt.group('month'))
    sd = dt.date(yr, month, 1)
    statsdtstr = str(sd.year) + "年" + str(sd.month) + "月"
    tcstr = "20" + mt.group('year') + "00" + mt.group('month') + mt.group('month')
    title_result['table_name'] = ws.title.strip()
    title_result['time_code'] = int(tcstr)
    title_result['stats_date'] = statsdtstr

start_row = 1 # formerly ws._print_rows.min_row
end_row = 3 # formerly ws._print_rows.max_row
rowriter = ws.iter_rows(min_row=valbounds[1], max_row=stubcr.min_row-1, min_col = valbounds[0], max_col=valbounds[2])
for row in rowriter:
    # Parse until we get the title
    for cell in row:
        if cell.value != None:
            m = TITLE_REGEX.match(cell.value)
            if m != None:
                title_result["title_population"] = m.group('title_population')
                if m.group('sub_population') != None:
                    title_result["sub_population"] = re.sub(r'[\s\n\u3000]' , '', m.group('sub_population'))
                else:
                    title_result["sub_population"] = '<NA>'
                title_result["cat01_label"] = m.group('cat01_label')
                title_result["cat02_label"] = m.group('cat02_label')
print(title_result)

# Find and parse the column header row
left_col = valbounds[0]
right_col = valbounds[2]
# col_headers = []
start_row = stubcr.min_row
end_row = stubcr.max_row
rowcount = end_row - start_row + 1
# make a nested list with rows being cells propagated across the header columns.
#  
# Iterate the columns and in each column iterate the rows
col_head = []
carriedcol = [None] * rowcount
# the stub column heads are for cat01. The rest are for cat02. 
cat01_levels = ['stub_lvl' + str(n) for n in range(rowcount)] # for stub header
cat02_levels = ['cat02lvl' + str(n) for n in range(rowcount)]
for col in ws.iter_cols(min_row=start_row, max_row=end_row, min_col=left_col, max_col=right_col):
    thiscolhead = []
    for idx, x in enumerate(col):
        if isinstance(x, Cell) and x.value != None:
            thiscolhead.append(x)
        else:
            thiscolhead.append(None)
    thiscolhead = carry_column_members(carriedcol.copy(), thiscolhead.copy(), 0)
    if len(col_head) <= stubcr.max_col - stubcr.min_col:
        col_head.append(list(zip(cat01_levels, thiscolhead.copy())))
    else:
        col_head.append(list(zip(cat02_levels, thiscolhead.copy())))
    carriedcol = thiscolhead.copy()

## from the col_head list, derive column headers.


$4:$6
'14-12-01-1'!$4:$6,'14-12-01-1'!$A:$B
4
6
{'table_name': '14-12-01-1', 'time_code': 2014001212, 'stats_date': '2014年12月', 'title_population': '在留外国人', 'sub_population': '<NA>', 'cat01_label': '国籍・地域', 'cat02_label': '在留資格（在留目的）'}


In [33]:
import re

col_heads = []
for h in col_head:
    #print(h)
    flist = [(a , re.sub(r'[\s\n\u3000]' , '', c.value)) if isinstance(c, Cell) else (a, '<NA>') for (a, c) in h]
    fdict = dict(flist)
    print(fdict)
    col_heads.append(fdict.copy())

{'stub_lvl0': '国籍・地域', 'stub_lvl1': '<NA>', 'stub_lvl2': '<NA>'}
{'stub_lvl0': '国籍・地域', 'stub_lvl1': '<NA>', 'stub_lvl2': '<NA>'}
{'cat02lvl0': '総数', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '教授', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '芸術', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '宗教', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '報道', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '投資・経営', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '法律・会計業務', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '医療', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '研究', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '教育', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '技術', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '人文知識・国際業務', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '企業内転勤', 'cat02lvl1': '<NA>', 'cat02lvl2': '<NA>'}
{'cat02lvl0': '興行', 'cat02lvl1': '<NA>', 'cat0

In [34]:
# Iterate the rows and in each row iterate the stub columns
start_col = stubcr.min_col
end_col = stubcr.max_col
colcount = end_col - start_col + 1

row_head = []
carriedrow = [None] * colcount
found_max_row = stubcr.max_row
# the stub column heads are for cat01. The rest are for cat02. 
cat01_levels = ['cat01lvl' + str(n) for n in range(colcount)]
for idxrow, row in enumerate(ws.iter_rows(min_row=stubcr.max_row +1, max_row=bounds[3], min_col=start_col, max_col=end_col)):
    thisrowhead = []
    hitcount = sum(1 for item in row if isinstance(item, Cell) and item.value != None)
    if hitcount == 0:
        break
    found_max_row += 1
    for idx, x in enumerate(row):
        if isinstance(x, Cell) and x.value != None:
            thisrowhead.append(x)
        else:
            thisrowhead.append(None)
    thisrowhead = carry_row_members(carriedrow.copy(), thisrowhead.copy(), 0)
    row_head.append(list(zip(cat01_levels, thisrowhead.copy())))
    carriedrow = thisrowhead.copy()

rowstubs = []
for h in row_head:
    rlist = [(a , re.sub(r'[\s\n\u3000]' , '', str(c.value))) if isinstance(c, Cell) else (a, '<NA>') for (a, c) in h]
    rdict = dict(rlist)
    print(rdict)
    rowstubs.append(rdict.copy())

{'cat01lvl0': '総数', 'cat01lvl1': '<NA>'}
{'cat01lvl0': 'アジア', 'cat01lvl1': '<NA>'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'アフガニスタン'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'アラブ首長国連邦'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'ミャンマー'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'バーレーン'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'ブータン'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'バングラデシュ'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'ブルネイ'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'カンボジア'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'スリランカ'}
{'cat01lvl0': 'アジア', 'cat01lvl1': '中国'}
{'cat01lvl0': 'アジア', 'cat01lvl1': '台湾'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'キプロス'}
{'cat01lvl0': 'アジア', 'cat01lvl1': '東ティモール'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'インド'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'インドネシア'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'イラン'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'イラク'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'イスラエル'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'ヨルダン'}
{'cat01lvl0': 'アジア', 'cat01lvl1': '韓国・朝鮮'}
{'cat01lvl0': 'アジア', 'cat01lvl1': 'クウェート'}
{'cat01lvl0': 'アジア', 

### Walk through the rows and build the dictionaries that will become the DataFrame

In [35]:
rows = []
min_allowed_count = found_max_col - stubcr.max_col # to check for end of table
for idxrow, row in enumerate(ws.iter_rows(min_row=stubcr.max_row +1, max_row=valbounds[3], min_col=start_col, max_col=found_max_col)):
    cell_count = sum(1 for item in row if isinstance(item, Cell) and item.value != None)
    if cell_count < min_allowed_count:
        print("cell_count: " + str(cell_count))
        break
    idxrow1 = idxrow + 1
    rowdict = {}
    rowdict.update(title_result)
    rowstub = rowstubs[idxrow]
    rowdict.update(rowstub)
    for idxcol, cell in enumerate(row):
        if idxcol >= stubcr.max_col:
            celldict = rowdict.copy()
            chead = col_heads[idxcol].copy()
            celldict.update(chead.copy())
            celldict['value'] = cell.value
            rows.append(celldict)
    
rows

[{'table_name': '14-12-01-1',
  'time_code': 2014001212,
  'stats_date': '2014年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01_label': '国籍・地域',
  'cat02_label': '在留資格（在留目的）',
  'cat01lvl0': '総数',
  'cat01lvl1': '<NA>',
  'cat02lvl0': '総数',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>',
  'value': 2121831},
 {'table_name': '14-12-01-1',
  'time_code': 2014001212,
  'stats_date': '2014年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01_label': '国籍・地域',
  'cat02_label': '在留資格（在留目的）',
  'cat01lvl0': '総数',
  'cat01lvl1': '<NA>',
  'cat02lvl0': '教授',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>',
  'value': 7565},
 {'table_name': '14-12-01-1',
  'time_code': 2014001212,
  'stats_date': '2014年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01_label': '国籍・地域',
  'cat02_label': '在留資格（在留目的）',
  'cat01lvl0': '総数',
  'cat01lvl1': '<NA>',
  'cat02lvl0': '芸術',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>',
  'value': 409},
 {'table

In [36]:
import pandas as pd
import csv

df = pd.DataFrame.from_dict(rows)

df.to_csv(processeddata, index=False, quoting=csv.QUOTE_ALL)
df

,table_name,time_code,stats_date,title_population,sub_population,cat01_label,cat02_label,cat01lvl0,cat01lvl1,cat02lvl0,cat02lvl1,cat02lvl2,value
0,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,総数,<NA>,総数,<NA>,<NA>,2121831
1,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,総数,<NA>,教授,<NA>,<NA>,7565
2,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,総数,<NA>,芸術,<NA>,<NA>,409
3,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,総数,<NA>,宗教,<NA>,<NA>,4528
4,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,総数,<NA>,報道,<NA>,<NA>,225
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8638,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,無国籍,<NA>,永住者,<NA>,<NA>,222
8639,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,無国籍,<NA>,日本人の配偶者等,<NA>,<NA>,54
8640,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,無国籍,<NA>,永住者の配偶者等,<NA>,<NA>,8
8641,14-12-01-1,2014001212,2014年12月,在留外国人,<NA>,国籍・地域,在留資格（在留目的）,無国籍,<NA>,定住者,<NA>,<NA>,141
